<a href="https://colab.research.google.com/github/ipreencekmr/anthropic/blob/main/Claude_With_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install anthropic python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 929.8/929.8 kB 12.5 MB/s eta 0:00:00


In [ ]:
#Load Environment Variable
import os
from google.colab import userdata

# Retrieve from Colab Secrets and set to environment variables
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

In [ ]:
#Create anthropic client
from anthropic import Anthropic
client = Anthropic()
model = 'claude-sonnet-4-6'

In [ ]:
#Make a request
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role":"user",
            "content":"What is quantum computing ? Answer in one sentence"
        }
    ]
)

In [ ]:
message.content[0].text

'Quantum computing is a type of computation that harnesses quantum mechanical phenomena such as **superposition** and **entanglement** to process information in ways that classical computers cannot, enabling the solving of certain complex problems much faster.'

# Multi Message Conversation

In [ ]:
#Helper functions to maintain history

def add_user_message(messages, text):
  user_message = {"role":"user", "content": text}
  messages.append(user_message)

def add_assistant_message(messages, text):
  assistant_message = {"role":"assistant", "content": text}
  messages.append(assistant_message)

def chat(messages):
  message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages
  )
  return message.content[0].text

In [ ]:
#Making a list of messages
conversation = []

#Add initial user question
add_user_message(conversation, "What is qunatum computing ? Answer in one sentence.")

print("What is qunatum computing ? Answer in one sentence.\n")

#Pass the list of conversation to the chat and get the answer
answer = chat(conversation)

print(answer)

#Add the answer as an assistant message to the conversation
add_assistant_message(conversation, answer)

#Ask follow up question
add_user_message(conversation, "Write another sentence")

print("Write another sentence\n")

#Invoke the chat to get the final answer
answer = chat(conversation)

print(answer)


What is qunatum computing ? Answer in one sentence.

**Quantum computing** is a type of computation that harnesses quantum mechanical phenomena, such as **superposition** and **entanglement**, to process information in ways that classical computers cannot efficiently replicate.
Write another sentence

Unlike classical computers that use bits (0s and 1s), quantum computers use **qubits**, which can exist in multiple states simultaneously, enabling them to solve complex problems — such as cryptography, drug discovery, and optimization — exponentially faster than traditional machines.


# Chat Bot Exercise

---



In [ ]:
messages = []

while True:
  user_input = input("> ")

  #add user message
  add_user_message(messages, user_input)

  #invoke claude to get the answer
  answer = chat(messages)

  #add answer to the conversation
  add_assistant_message(messages, answer)

  #print the answer
  print("---")
  print(answer)
  print("---")

> what is 1+1
---
1 + 1 = **2**
---
> and 2+6
---
2 + 6 = **8**
---
> multiply last two results
---
2 × 8 = **16**
---


KeyboardInterrupt: Interrupted by user

# Math tutor with a system prompt

In [ ]:
def chat(messages, system=None, temperature=1.0):

  params = {
    "model":model,
    "max_tokens":1000,
    "messages":messages,
    "temperature":temperature
  }

  if system:
    params["system"] = system

  message = client.messages.create(**params)
  return message.content[0].text

In [ ]:
  system = """
    You are a patient math tutor.
    Do not directly answer a student's questions.
    Guide them to a solution step by step.
  """

  message = []

  add_user_message(messages, "How do I solve 5x+3=2, what is the value of x ?")

  answer = chat(messages, system)

  print(answer)

Let's work through this step by step! 😊

**Step 1:** We need to get the variable (x) by itself. First, let's get rid of the **+3** on the left side.

What do you think we should do to both sides of the equation to remove the +3?


# Code Writer

In [ ]:
messages = []

add_user_message(messages, "Write a python function that checks a string for duplicate characters.")

answer = chat(messages, system="You are a Python engineer who writes a very concise code")

print(answer)

```python
def has_duplicates(s):
    return len(s) != len(set(s))
```

**Examples:**
```python
print(has_duplicates("hello"))   # True  ('l' appears twice)
print(has_duplicates("world"))   # False
print(has_duplicates("python"))  # False
print(has_duplicates("apple"))   # True  ('p' appears twice)
```


# Get a movie idea with high creativity (temp value high)

- Temperature controls randomness of the result
- Less temperature means more deterministic output
- High temperature means more randomness and creativity

In [ ]:
messages = []

add_user_message(messages, "Generate a movie idea in one sentence")

answer = chat(messages, temperature=1.0)

print(answer)

Here's a movie idea:

**A retired safecracker with early-onset Alzheimer's must pull off one final heist before he forgets the combination to the vault that holds the only evidence to free his wrongfully imprisoned son.**


# Streaming

In [ ]:
messages = []

add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    max_tokens=1000,
    model=model,
    messages=messages,
    stream=True
)

for event in stream:
  print(event)

RawMessageStartEvent(message=Message(id='msg_01AC6ZxCvFifJB1RcWccVVes', container=None, content=[], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=18, output_tokens=1, output_tokens_details=None, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='Here', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' is a one sentence description of a fake database:\n\n**"NovaBase', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='"** is a

In [ ]:
messages = []

add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(model=model, max_tokens=1000, messages=messages) as stream:
  for text in stream.text_stream:
    print(text, end="")

"CustomerVault is a fictional relational database storing fabricated customer profiles, including made-up names, addresses, and purchase histories for the non-existent retail company, ShopWorld Inc."

In [ ]:
stream.get_final_message()

ParsedMessage(id='msg_01C69ESyDyRUR5PyQKQQZJBp', container=None, content=[ParsedTextBlock(citations=None, text='"CustomerVault is a fictional relational database storing fabricated customer profiles, including made-up names, addresses, and purchase histories for the non-existent retail company, ShopWorld Inc."', type='text', parsed_output=None)], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=18, output_tokens=44, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

# Structured data

In [ ]:
def chat(messages, system=None, temperature=1.0, stop_sequences=[], output_config=None):

  params = {
    "model":"claude-sonnet-4-5-20250929",
    "max_tokens":1000,
    "messages":messages,
    "temperature":temperature,
    "stop_sequences":stop_sequences
  }

  if system:
    params["system"] = system

  if output_config:
    params["output_config"] = output_config

  message = client.messages.create(**params)
  return message.content[0].text

In [ ]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")

text = chat(messages)

text

'Here\'s a simple EventBridge rule in JSON:\n\n```json\n{\n  "source": ["aws.ec2"],\n  "detail-type": ["EC2 Instance State-change Notification"],\n  "detail": {\n    "state": ["stopped"]\n  }\n}\n```\n\nThis rule triggers when an **EC2 instance is stopped**.'

Here's a simple EventBridge rule in JSON:

```json
{
  "source": ["aws.ec2"],
  "detail-type": ["EC2 Instance State-change Notification"],
  "detail": {
    "state": ["stopped"]
  }
}
```

This rule triggers when an **EC2 instance is stopped**.

### Control the output format

In [ ]:
event_pattern_schema = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "source": {
            "type": "array",
            "items": {"type": "string"}
        },
        "detail-type": {
            "type": "array",
            "items": {"type": "string"}
        },
        "detail": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "state": {
                    "type": "array",
                    "items": {"type": "string"}
                }
            },
            "required": ["state"]
        }
    },
    "required": ["source", "detail-type", "detail"]
}

In [ ]:
output_config = {
    "format": {
        "type": "json_schema",
        "schema": eventbridge_schema
      }
    }

messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")

text = chat(messages, output_config=output_config)

text

'{"Name":"MyRule","EventPattern":{"source":["aws.ec2"],"detail-type":["EC2 Instance State-change Notification"]},"State":"ENABLED","Targets":[{"Id":"MyTarget","Arn":"arn:aws:lambda:us-east-1:123456789012:function:MyFunction"}]}'

In [ ]:
import json

# Clean up and parse the JSON
clean_json = json.loads(text.strip())

In [ ]:
clean_json

{'Name': 'MyRule',
 'EventPattern': {'source': ['aws.ec2'],
  'detail-type': ['EC2 Instance State-change Notification']},
 'State': 'ENABLED',
 'Targets': [{'Id': 'MyTarget',
   'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:MyFunction'}]}

# Exercise to remove Message Prefills
- Get all three different commands in a single response
- There shoudn't be explainations or comments

In [ ]:
messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short
"""

add_user_message(messages, prompt)

text = chat(messages)

text.strip()

'Here are three short AWS CLI commands:\n\n1. **List S3 Buckets**\n```bash\naws s3 ls\n```\n\n2. **Describe EC2 Instances**\n```bash\naws ec2 describe-instances\n```\n\n3. **List IAM Users**\n```bash\naws iam list-users\n```'

In [ ]:
from IPython.display import Markdown
Markdown(text)

Here are three short AWS CLI commands:

1. **List S3 Buckets**
```bash
aws s3 ls
```

2. **Describe EC2 Instances**
```bash
aws ec2 describe-instances
```

3. **List IAM Users**
```bash
aws iam list-users
```

In [ ]:
from IPython.core.display import Markdown
messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short
"""

add_user_message(messages, prompt)

add_assistant_message(messages, "```bash")

text = chat(messages, stop_sequences=["```"])

text.strip()

Markdown(text)


aws s3 ls

aws ec2 describe-instances --region us-east-1

aws iam list-users
